In [1]:
import pandas as pd
import os
df = pd.read_csv("../0_data/1_cleaned/fact_transactions.csv")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# 1. Calculate the Snapshot Date as One Day After the Latest Invoice Date in the Dataset
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

print(snapshot_date)

2011-12-10 12:50:00


In [2]:
# 2. Calculate RFM Metrics for Each Customer
rfm = (
    df.groupby("Customer ID")
      .agg({
          "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
          "Invoice": "nunique",
          "Revenue": "sum"
      })
      .reset_index()
)

rfm.columns = [
    "CustomerID",
    "Recency",
    "Frequency",
    "Monetary"
]

rfm["CustomerID"] = rfm["CustomerID"].astype(int)
rfm["Recency"] = rfm["Recency"].astype(int)
rfm["Frequency"] = rfm["Frequency"].astype(int)

# rfm.head()
rfm.describe()

,CustomerID,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,201.331916,6.289384,2955.904095
std,1715.572666,209.338707,13.009406,14440.852688
min,12346.000000,1.000000,1.000000,2.950000
25%,13833.250000,26.000000,1.000000,342.280000
50%,15314.500000,96.000000,3.000000,867.740000
75%,16797.750000,380.000000,7.000000,2248.305000
max,18287.000000,739.000000,398.000000,580987.040000


In [3]:
# 3. Assign RFM Scores to Each Customer
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    q=5,
    labels=[5, 4, 3, 2, 1]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    q=5,
    labels=[1, 2, 3, 4, 5]
)

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

rfm["RFM_Total"] = (
    rfm["R_Score"].astype(int)
    + rfm["F_Score"].astype(int)
    + rfm["M_Score"].astype(int)
)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total
0,12346,326,12,77556.46,2,5,5,255,12
1,12347,2,8,4921.53,5,4,5,545,14
2,12348,75,5,2019.40,3,4,4,344,11
3,12349,19,4,4428.69,5,3,5,535,13
4,12350,310,1,334.40,2,1,2,212,5


In [4]:
# 4. Segment Customers Based on RFM Scores
def segment_customer(score):
    if score >= 13:
        return "Champions"
    elif score >= 10:
        return "Loyal Customers"
    elif score >= 8:
        return "Potential Loyalists"
    elif score >= 5:
        return "At Risk"
    else:
        return "Lost Customers"

rfm["Segment"] = rfm["RFM_Total"].apply(segment_customer)

# rfm
# rfm["Segment"].value_counts()

os.makedirs("../0_data/1_cleaned", exist_ok=True)
rfm.to_csv("../0_data/1_cleaned/dim_customer_rfm.csv", index=False)

In [5]:
# 5. Segment Summarizing
segment_summary = (
    rfm.groupby("Segment")
       .agg({
           "CustomerID": "count",
           "Monetary": "sum",
           "Frequency": "median",
           "Recency": "median"
       })
       .round(2)
       .sort_values("Monetary", ascending=False)
)

segment_summary

,CustomerID,Monetary,Frequency,Recency
Segment,,,,
Champions,1295,12533229.88,12.0,16.0
Loyal Customers,1357,3070956.81,5.0,54.0
Potential Loyalists,980,928744.85,3.0,122.0
At Risk,1440,683302.85,1.0,298.5
Lost Customers,806,158569.88,1.0,523.0
